# Utilitários para imagens RAW do XRMC

Este notebook fornece as funções utilizadas por `generateSino.ipynb`:

- `readRawXRMCImg`: lê uma imagem RAW do XRMC;
- `readRawXRMCFolder`: lê uma sequência de projeções e cria um volume 3D;
- `printImgData`: mostra dimensões, valores mínimo/máximo/médio e tipo.

Carregue as funções no notebook principal com:

```python
%run ./utils.ipynb
```


In [ ]:
from pathlib import Path
from typing import Union

import numpy as np

PathLike = Union[str, Path]


In [ ]:
def readRawXRMCImg(
    fileName: PathLike,
    rows: int,
    cols: int,
    dataType: str | np.dtype = "float64",
    offset: int = 0,
) -> np.ndarray:
    """Lê uma imagem RAW produzida pelo XRMC.

    Parameters
    ----------
    fileName : str ou pathlib.Path
        Caminho do arquivo RAW.
    rows, cols : int
        Número de linhas e colunas da imagem.
    dataType : str ou numpy.dtype
        Tipo numérico armazenado no arquivo, por exemplo ``float64``.
    offset : int
        Número de bytes a ignorar no início do arquivo.

    Returns
    -------
    numpy.ndarray
        Matriz com formato ``(rows, cols)``.
    """
    path = Path(fileName).expanduser()

    if not path.is_file():
        raise FileNotFoundError(f"Arquivo RAW não encontrado: {path.resolve()}")

    if not isinstance(rows, int) or not isinstance(cols, int):
        raise TypeError("rows e cols precisam ser números inteiros.")

    if rows <= 0 or cols <= 0:
        raise ValueError("rows e cols precisam ser maiores que zero.")

    if not isinstance(offset, int) or offset < 0:
        raise ValueError("offset precisa ser um inteiro maior ou igual a zero.")

    try:
        dtype = np.dtype(dataType)
    except TypeError as error:
        raise TypeError(f"Tipo de dado inválido: {dataType}") from error

    pixel_count = rows * cols
    expected_data_bytes = pixel_count * dtype.itemsize
    file_size = path.stat().st_size
    available_bytes = file_size - offset

    if available_bytes < expected_data_bytes:
        raise ValueError(
            f"O arquivo {path.name} é menor do que o esperado. "
            f"Necessário após o offset: {expected_data_bytes} bytes; "
            f"disponível: {max(available_bytes, 0)} bytes. "
            "Confira rows, cols, dataType e offset."
        )

    if available_bytes != expected_data_bytes:
        extra = available_bytes - expected_data_bytes
        print(
            f"Aviso: {path.name} possui {extra} byte(s) além da imagem esperada. "
            "Confira se o offset está correto."
        )

    data = np.fromfile(
        path,
        dtype=dtype,
        count=pixel_count,
        offset=offset,
    )

    if data.size != pixel_count:
        raise ValueError(
            f"Não foi possível ler todos os pixels de {path.name}: "
            f"esperados {pixel_count}, lidos {data.size}."
        )

    return data.reshape((rows, cols))


In [ ]:
def readRawXRMCFolder(
    folderName: PathLike,
    filePrefix: str,
    startIndex: int,
    endIndex: int,
    indexStep: int,
    rows: int,
    cols: int,
    dataType: str | np.dtype = "float64",
    offset: int = 0,
) -> np.ndarray:
    """Lê uma sequência de projeções RAW do XRMC.

    Os arquivos devem seguir o padrão ``prefixo + índice com 4 dígitos + .dat``.
    Por exemplo: ``img_0000.dat``, ``img_0001.dat`` e assim por diante.

    O índice final é exclusivo, como ocorre em ``range``. Assim,
    ``startIndex=0`` e ``endIndex=180`` leem 180 arquivos, de 0 a 179.

    Returns
    -------
    numpy.ndarray
        Volume com formato ``(rows, cols, número_de_projeções)``.
    """
    folder = Path(folderName).expanduser()

    if not folder.is_dir():
        raise NotADirectoryError(
            f"Pasta das projeções não encontrada: {folder.resolve()}"
        )

    if not all(isinstance(value, int) for value in (startIndex, endIndex, indexStep)):
        raise TypeError("startIndex, endIndex e indexStep precisam ser inteiros.")

    if indexStep == 0:
        raise ValueError("indexStep não pode ser zero.")

    indices = list(range(startIndex, endIndex, indexStep))
    if not indices:
        raise ValueError("O intervalo informado não contém nenhuma projeção.")

    projections = []
    missing_files = []

    for index in indices:
        file_path = folder / f"{filePrefix}{index:04d}.dat"

        if not file_path.is_file():
            missing_files.append(file_path.name)
            continue

        projection = readRawXRMCImg(
            file_path,
            rows,
            cols,
            dataType,
            offset,
        )
        projections.append(projection)

    if missing_files:
        preview = ", ".join(missing_files[:10])
        remaining = len(missing_files) - 10
        suffix = f" e mais {remaining}" if remaining > 0 else ""
        raise FileNotFoundError(
            f"Faltam {len(missing_files)} arquivo(s) na pasta {folder.resolve()}: "
            f"{preview}{suffix}."
        )

    return np.stack(projections, axis=2)


In [ ]:
def printImgData(image: np.ndarray, imageName: str = "image") -> None:
    """Imprime informações estatísticas de uma imagem ou volume."""
    array = np.asarray(image)

    if array.size == 0:
        raise ValueError("A imagem está vazia.")

    finite_values = array[np.isfinite(array)]

    if finite_values.size == 0:
        minimum = maximum = mean = float("nan")
    else:
        minimum = finite_values.min()
        maximum = finite_values.max()
        mean = finite_values.mean()

    print(
        f"{imageName}:: Shape:{array.shape}, "
        f"Min:{minimum:.4e}, Max:{maximum:.4e}, "
        f"Mean:{mean:.4e}, Type:{array.dtype}"
    )


In [ ]:
def plotFourProjs(
    projs: np.ndarray,
    _prjs=(0, 1, 2, 3),
    _minv=None,
    _maxv=None,
    cmap="gray",
    figsize=(12, 10),
) -> None:
    """Mostra quatro projeções em uma grade 2 x 2.

    A função espera que ``projs`` tenha o formato:
        (número_de_projeções, linhas, colunas)

    Esse é o formato obtido após:
        projs = projs.transpose(2, 0, 1)
    """
    projections = np.asarray(projs)

    if projections.ndim != 3:
        raise ValueError(
            "projs precisa ser um array 3D no formato "
            "(número_de_projeções, linhas, colunas). "
            f"Formato recebido: {projections.shape}"
        )

    indices = list(_prjs)

    if len(indices) != 4:
        raise ValueError(
            f"_prjs precisa conter exatamente quatro índices; recebeu {len(indices)}."
        )

    invalid = [
        index
        for index in indices
        if not isinstance(index, (int, np.integer))
        or index < 0
        or index >= projections.shape[0]
    ]

    if invalid:
        raise IndexError(
            f"Índice(s) inválido(s): {invalid}. "
            f"O volume possui {projections.shape[0]} projeções, "
            f"com índices válidos de 0 a {projections.shape[0] - 1}."
        )

    fig, axes = plt.subplots(2, 2, figsize=figsize)
    axes = axes.ravel()

    images = []

    for axis, projection_index in zip(axes, indices):
        image = axis.imshow(
            projections[projection_index],
            cmap=cmap,
            vmin=_minv,
            vmax=_maxv,
        )
        axis.set_title(f"Projeção {projection_index}")
        axis.axis("off")
        images.append(image)

    if _minv is not None or _maxv is not None:
        fig.colorbar(
            images[-1],
            ax=axes.tolist(),
            shrink=0.8,
            label="Intensidade",
        )

    fig.tight_layout()
    plt.show()


## Exemplo de uso

```python
folderName = './imagens_raw'

flat = readRawXRMCImg(
    f'{folderName}/flat.dat',
    1200,
    1400,
    'float64',
    0,
)

projs = readRawXRMCFolder(
    folderName,
    'img_',
    0,
    180,
    1,
    1200,
    1400,
    'float64',
    0,
)

printImgData(flat, 'flat')
printImgData(projs, 'projection')
```
